<a href="https://colab.research.google.com/github/huhan2514/CampusGear-Online-Bookstore-and-Stationery-Store/blob/main/CampusGear_Online_Bookstore_and_Stationery_Store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 2：Installation Environment

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu pandas numpy scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.4 MB/s eta 0:00:00


# Cell 3：Import required libraries

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Cell 4：Set random seed and device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

Current device: cpu


# Cell 5:Define basic project information(定义项目基本信息)

In [ ]:
PROJECT_TITLE = "Development of an LLM, RAG, and Fine-Tuned Customer Service Chatbot for CampusGear Online Bookstore and Stationery Store"

COMPANY_NAME = "CampusGear Online Bookstore and Stationery Store"

print(PROJECT_TITLE)
print("Company:", COMPANY_NAME)

Development of an LLM, RAG, and Fine-Tuned Customer Service Chatbot for CampusGear Online Bookstore and Stationery Store
Company: CampusGear Online Bookstore and Stationery Store


# Cell 6：Read product-related data from the same Olist dataset

In [ ]:
# Cell 6: Load Product-Related Data from Olist Cloud Dataset

# Olist product-related datasets
products_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_products_dataset.csv"
category_translation_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/product_category_name_translation.csv"
order_items_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_items_dataset.csv"

raw_products_df = pd.read_csv(products_url)
category_translation_df = pd.read_csv(category_translation_url)
raw_order_items_df = pd.read_csv(order_items_url)

print("Raw products dataset loaded successfully.")
print("Products shape:", raw_products_df.shape)

print("\nProduct category translation dataset loaded successfully.")
print("Category translation shape:", category_translation_df.shape)

print("\nOrder items dataset loaded successfully.")
print("Order items shape:", raw_order_items_df.shape)

print("\nProducts columns:")
print(raw_products_df.columns.tolist())

print("\nOrder items columns:")
print(raw_order_items_df.columns.tolist())

raw_products_df.head()

Raw products dataset loaded successfully.
Products shape: (32951, 9)

Product category translation dataset loaded successfully.
Category translation shape: (71, 2)

Order items dataset loaded successfully.
Order items shape: (112650, 7)

Products columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Order items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


# Cell 6.1：Convert Olist product data into CampusGear product database

In [ ]:
# Cell 6.1: Transform Olist Products into CampusGear Product Database

# Merge product category with English translation
products_with_category = raw_products_df.merge(
    category_translation_df,
    on="product_category_name",
    how="left"
)

# Merge products with order items to obtain price and freight information
product_order_info = raw_order_items_df.groupby("product_id").agg(
    average_price_rm=("price", "mean"),
    average_freight_rm=("freight_value", "mean"),
    total_sold=("order_id", "count")
).reset_index()

products_source_df = products_with_category.merge(
    product_order_info,
    on="product_id",
    how="inner"
)

# Keep only products with useful category information
products_source_df = products_source_df.dropna(
    subset=["product_category_name_english", "average_price_rm"]
)

# Select categories suitable for CampusGear bookstore and stationery store
campusgear_categories = [
    "books_general_interest",
    "books_technical",
    "books_imported",
    "stationery",
    "office_furniture",
    "computers_accessories",
    "electronics"
]

products_source_df = products_source_df[
    products_source_df["product_category_name_english"].isin(campusgear_categories)
].copy()

# Sort by popularity and price
products_source_df = products_source_df.sort_values(
    by=["total_sold", "average_price_rm"],
    ascending=False
).head(40)

products_source_df = products_source_df.reset_index(drop=True)

# Convert category names into cleaner display names
def clean_category_name(category):
    return category.replace("_", " ").title()

# Create readable product names based on category
def create_product_name(row, index):
    category = clean_category_name(row["product_category_name_english"])

    if "Books Technical" in category:
        return f"CampusGear Technical Book {index + 1}"
    elif "Books General Interest" in category:
        return f"CampusGear General Reading Book {index + 1}"
    elif "Books Imported" in category:
        return f"CampusGear Imported Book {index + 1}"
    elif "Stationery" in category:
        return f"CampusGear Stationery Product {index + 1}"
    elif "Computers Accessories" in category:
        return f"CampusGear Computer Accessory {index + 1}"
    elif "Electronics" in category:
        return f"CampusGear Electronic Study Tool {index + 1}"
    elif "Office Furniture" in category:
        return f"CampusGear Study Desk Accessory {index + 1}"
    else:
        return f"CampusGear Product {index + 1}"

products_df = pd.DataFrame()

products_df["product_id"] = [f"P{str(i + 1).zfill(4)}" for i in range(len(products_source_df))]
products_df["original_product_id"] = products_source_df["product_id"]
products_df["product_name"] = [
    create_product_name(row, i)
    for i, row in products_source_df.iterrows()
]
products_df["category"] = products_source_df["product_category_name_english"].apply(clean_category_name)
products_df["price_rm"] = products_source_df["average_price_rm"].round(2)
products_df["freight_rm"] = products_source_df["average_freight_rm"].round(2)
products_df["total_sold"] = products_source_df["total_sold"]

products_df["description"] = (
    "A CampusGear product from the " + products_df["category"] +
    " category. This product is suitable for students, academic study, office work, or learning support. " +
    "The product information is derived from the Olist e-commerce product and order item dataset."
)

# Save product ID mapping for order database
product_id_mapping = dict(zip(products_df["original_product_id"], products_df["product_id"]))

print("CampusGear product database created from Olist dataset.")
print("Final product database shape:", products_df.shape)

products_df.head(15)

CampusGear product database created from Olist dataset.
Final product database shape: (40, 8)


,product_id,original_product_id,product_name,category,price_rm,freight_rm,total_sold,description
0,P0001,d1c427060a0f73f6b889a5c7c61f2ac4,CampusGear Computer Accessory 1,Computers Accessories,137.65,40.12,343,A CampusGear product from the Computers Access...
1,P0002,3dd2a17168ec895c781a9191c1e95ad7,CampusGear Computer Accessory 2,Computers Accessories,149.94,26.02,274,A CampusGear product from the Computers Access...
2,P0003,e53e557d5a159f5aa2c5e995dfdf244b,CampusGear Computer Accessory 3,Computers Accessories,84.37,16.23,183,A CampusGear product from the Computers Access...
3,P0004,656e0eca68dcecf6a31b8ececfabe3e8,CampusGear Computer Accessory 4,Computers Accessories,88.49,15.09,141,A CampusGear product from the Computers Access...
4,P0005,36f60d45225e60c7da4558b070ce4b60,CampusGear Computer Accessory 5,Computers Accessories,88.51,12.42,127,A CampusGear product from the Computers Access...
5,P0006,7ce94ab189134e2d3c05f496d635419c,CampusGear Electronic Study Tool 6,Electronics,13.65,15.68,106,A CampusGear product from the Electronics cate...
6,P0007,d5991653e037ccb7af6ed7d94246b249,CampusGear Computer Accessory 7,Computers Accessories,146.72,28.73,103,A CampusGear product from the Computers Access...
7,P0008,3f14d740544f37ece8a9e7bc8349797e,CampusGear Computer Accessory 8,Computers Accessories,84.96,16.82,91,A CampusGear product from the Computers Access...
8,P0009,4298b7e67dc399c200662b569563a2b2,CampusGear Computer Accessory 9,Computers Accessories,141.81,18.65,88,A CampusGear product from the Computers Access...
9,P0010,dbb67791e405873b259e4656bf971246,CampusGear Computer Accessory 10,Computers Accessories,79.27,13.36,88,A CampusGear product from the Computers Access...


# Cell 7：Read order, payment, and delivery data from the same Olist dataset.

In [ ]:
# Cell 7: Load Order, Payment, and Delivery Data from Olist Cloud Dataset

orders_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_orders_dataset.csv"
payments_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_payments_dataset.csv"

raw_orders_df = pd.read_csv(orders_url)
raw_payments_df = pd.read_csv(payments_url)

print("Raw orders dataset loaded successfully.")
print("Orders shape:", raw_orders_df.shape)

print("\nRaw payments dataset loaded successfully.")
print("Payments shape:", raw_payments_df.shape)

print("\nOrders columns:")
print(raw_orders_df.columns.tolist())

print("\nPayments columns:")
print(raw_payments_df.columns.tolist())

raw_orders_df.head()

Raw orders dataset loaded successfully.
Orders shape: (99441, 8)

Raw payments dataset loaded successfully.
Payments shape: (103886, 5)

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Payments columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


# Cell 7.1：Convert to CampusGear order database

In [ ]:
# Cell 7.1: Create CampusGear Order Database from the Same Olist Dataset

import hashlib

# Use only order items whose products are included in CampusGear product database
campusgear_order_items = raw_order_items_df[
    raw_order_items_df["product_id"].isin(product_id_mapping.keys())
].copy()

# Add CampusGear product ID
campusgear_order_items["campusgear_product_id"] = campusgear_order_items["product_id"].map(product_id_mapping)

# Summarize order items
order_item_summary = campusgear_order_items.groupby("order_id").agg(
    original_product_id=("product_id", "first"),
    campusgear_product_id=("campusgear_product_id", "first"),
    item_count=("order_item_id", "count"),
    item_price_rm=("price", "sum"),
    freight_value_rm=("freight_value", "sum")
).reset_index()

# Summarize payment information
payment_summary = raw_payments_df.groupby("order_id").agg(
    payment_method=("payment_type", "first"),
    payment_value_rm=("payment_value", "sum")
).reset_index()

# Merge orders + item summary + payment summary
orders_source_df = raw_orders_df.merge(
    order_item_summary,
    on="order_id",
    how="inner"
).merge(
    payment_summary,
    on="order_id",
    how="left"
)

# Choose useful statuses for testing
selected_statuses = ["delivered", "shipped", "processing", "canceled", "unavailable"]

sample_list = []

for status in selected_statuses:
    subset = orders_source_df[orders_source_df["order_status"] == status].copy()

    if len(subset) > 0:
        sample_size = min(5, len(subset))
        sample_list.append(subset.sample(n=sample_size, random_state=SEED))

orders_sample = pd.concat(sample_list, ignore_index=True)
orders_sample = orders_sample.reset_index(drop=True)

# Create tracking number from original order ID
def create_tracking_number(original_order_id):
    hashed_value = hashlib.md5(original_order_id.encode()).hexdigest()
    number = int(hashed_value[:8], 16) % 900000000 + 100000000
    return "MY" + str(number)

# Format date
def format_date(date_value):
    if pd.isna(date_value):
        return "Not Available"
    try:
        return pd.to_datetime(date_value).strftime("%d %B %Y")
    except:
        return "Not Available"

# Get product name from CampusGear product database
def get_product_name(campusgear_product_id):
    product_row = products_df[products_df["product_id"] == campusgear_product_id]

    if product_row.empty:
        return "CampusGear Product"
    else:
        return product_row.iloc[0]["product_name"]

orders_df = pd.DataFrame()

orders_df["order_id"] = [f"CG{1001 + i}" for i in range(len(orders_sample))]
orders_df["original_order_id"] = orders_sample["order_id"]
orders_df["customer_ref"] = orders_sample["customer_id"].apply(lambda x: "Customer_" + str(x)[:6])
orders_df["status"] = orders_sample["order_status"].str.title()

orders_df["product_id"] = orders_sample["campusgear_product_id"]
orders_df["product_name"] = orders_sample["campusgear_product_id"].apply(get_product_name)
orders_df["item_count"] = orders_sample["item_count"]

orders_df["tracking_no"] = orders_sample.apply(
    lambda row: "Not Available"
    if row["order_status"] in ["canceled", "unavailable", "processing"]
    else create_tracking_number(row["order_id"]),
    axis=1
)

orders_df["purchase_date"] = orders_sample["order_purchase_timestamp"].apply(format_date)
orders_df["approved_date"] = orders_sample["order_approved_at"].apply(format_date)
orders_df["carrier_delivery_date"] = orders_sample["order_delivered_carrier_date"].apply(format_date)
orders_df["customer_delivery_date"] = orders_sample["order_delivered_customer_date"].apply(format_date)
orders_df["estimated_delivery_date"] = orders_sample["order_estimated_delivery_date"].apply(format_date)

orders_df["payment_method"] = orders_sample["payment_method"].fillna("Not Available").str.title()
orders_df["payment_value_rm"] = orders_sample["payment_value_rm"].fillna(0).round(2)
orders_df["item_price_rm"] = orders_sample["item_price_rm"].fillna(0).round(2)
orders_df["freight_value_rm"] = orders_sample["freight_value_rm"].fillna(0).round(2)

# Save order ID mapping for refund database
order_id_mapping = dict(zip(orders_df["original_order_id"], orders_df["order_id"]))

print("CampusGear order database created from the same Olist dataset.")
print("Final order database shape:", orders_df.shape)

orders_df.head(15)

CampusGear order database created from the same Olist dataset.
Final order database shape: (15, 17)


,order_id,original_order_id,customer_ref,status,product_id,product_name,item_count,tracking_no,purchase_date,approved_date,carrier_delivery_date,customer_delivery_date,estimated_delivery_date,payment_method,payment_value_rm,item_price_rm,freight_value_rm
0,CG1001,219fda9e83f9a3ac8e1baa10d6cc6366,Customer_9e7a24,Delivered,P0034,CampusGear Computer Accessory 34,1,MY558017943,08 June 2017,10 June 2017,13 June 2017,14 June 2017,22 June 2017,Boleto,40.01,31.90,8.11
1,CG1002,7e7c6416e3aece73d5dda3fc6f33a044,Customer_a3ba7c,Delivered,P0016,CampusGear Computer Accessory 16,1,MY639487529,31 March 2018,31 March 2018,04 April 2018,10 April 2018,24 April 2018,Credit_Card,90.38,72.00,18.38
2,CG1003,2c88b5879d666444b90643a3817f9b97,Customer_a7f913,Delivered,P0027,CampusGear Study Desk Accessory 27,1,MY689726772,17 March 2017,17 March 2017,28 March 2017,04 April 2017,17 April 2017,Voucher,134.61,119.60,15.01
3,CG1004,63455a963c823380655caf4dc3ba5397,Customer_04ee08,Delivered,P0019,CampusGear Electronic Study Tool 19,1,MY337469262,11 January 2018,11 January 2018,13 January 2018,19 January 2018,30 January 2018,Credit_Card,20.03,12.25,7.78
4,CG1005,68045e87308a14e1d77f14ff287a2b3b,Customer_5d3580,Delivered,P0038,CampusGear Electronic Study Tool 38,1,MY282300367,26 March 2018,28 March 2018,29 March 2018,04 April 2018,06 April 2018,Boleto,30.19,21.90,8.29
5,CG1006,31c48fb282a0daf35c1aceadc0231bc1,Customer_30c61e,Shipped,P0005,CampusGear Computer Accessory 5,1,MY389711821,23 January 2018,23 January 2018,26 January 2018,Not Available,16 February 2018,Credit_Card,104.87,89.49,15.38
6,CG1007,0538bda829ac14e64a201dc84fb1ba3b,Customer_84cb48,Shipped,P0029,CampusGear Study Desk Accessory 29,1,MY765406921,18 May 2018,21 May 2018,06 June 2018,Not Available,25 June 2018,Credit_Card,224.69,169.99,54.70
7,CG1008,6e77d9428b5dec0e2557b439ea6d61df,Customer_567c38,Shipped,P0001,CampusGear Computer Accessory 1,2,MY529308756,19 April 2018,19 April 2018,19 April 2018,Not Available,15 May 2018,Credit_Card,277.48,238.00,39.48
8,CG1009,ea3d171a91e27a6f523ad45abb5dcf20,Customer_8d602b,Shipped,P0013,CampusGear Computer Accessory 13,1,MY798697117,11 April 2018,11 April 2018,12 April 2018,Not Available,03 May 2018,Credit_Card,189.56,180.00,9.56
9,CG1010,88743d1ed8e7b6ca01e12830855ccda2,Customer_6ab3e2,Shipped,P0008,CampusGear Computer Accessory 8,1,MY409921345,28 February 2018,28 February 2018,02 March 2018,Not Available,22 March 2018,Credit_Card,103.04,86.00,17.04


# Cell 8：Read comment data from the same Olist dataset

In [ ]:
# Cell 8: Load Review Dataset from the Same Olist Dataset

reviews_url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_reviews_dataset.csv"

raw_reviews_df = pd.read_csv(reviews_url)

print("Raw reviews dataset loaded successfully.")
print("Reviews shape:", raw_reviews_df.shape)

print("\nReview dataset columns:")
print(raw_reviews_df.columns.tolist())

raw_reviews_df.head()

Raw reviews dataset loaded successfully.
Reviews shape: (99224, 7)

Review dataset columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


# Cell 8.1：Generate refund database from the same Olist dataset

In [ ]:
# Cell 8.1: Create Refund Database from the Same Olist Dataset

# Merge orders sample with reviews
refund_source_df = orders_sample.merge(
    raw_reviews_df[["order_id", "review_score", "review_comment_message"]],
    on="order_id",
    how="left"
)

# Merge with payment summary
refund_source_df = refund_source_df.merge(
    payment_summary,
    on="order_id",
    how="left",
    suffixes=("", "_payment")
)

# Select refund-related cases from the same sampled CampusGear orders
refund_candidates = refund_source_df[
    (refund_source_df["order_status"].isin(["canceled", "unavailable"])) |
    (refund_source_df["review_score"].fillna(5) <= 2)
].copy()

# If there are not enough refund cases from sampled orders, get more from the same Olist dataset
if len(refund_candidates) < 5:
    extra_refund_source = raw_orders_df.merge(
        order_item_summary,
        on="order_id",
        how="inner"
    ).merge(
        raw_reviews_df[["order_id", "review_score", "review_comment_message"]],
        on="order_id",
        how="left"
    ).merge(
        payment_summary,
        on="order_id",
        how="left"
    )

    extra_candidates = extra_refund_source[
        (extra_refund_source["order_status"].isin(["canceled", "unavailable"])) |
        (extra_refund_source["review_score"].fillna(5) <= 2)
    ].copy()

    extra_candidates = extra_candidates.sample(
        n=min(10, len(extra_candidates)),
        random_state=SEED
    )

    refund_candidates = pd.concat(
        [refund_candidates, extra_candidates],
        ignore_index=True
    )

refund_candidates = refund_candidates.drop_duplicates(subset=["order_id"]).head(15)
refund_candidates = refund_candidates.reset_index(drop=True)

# Create CampusGear-style order ID
def get_campusgear_order_id(original_order_id, index):
    if original_order_id in order_id_mapping:
        return order_id_mapping[original_order_id]
    else:
        return f"CG{3001 + index}"

# Generate refund reason
def generate_refund_reason(row):
    status = row["order_status"]
    review_score = row.get("review_score", np.nan)

    if status == "canceled":
        return "Order was cancelled"
    elif status == "unavailable":
        return "Product was unavailable"
    elif not pd.isna(review_score) and review_score <= 2:
        return "Customer complaint based on low review score"
    else:
        return "Refund-related customer service case"

# Generate refund status
def generate_refund_status(row):
    status = row["order_status"]

    if status == "canceled":
        return "Completed"
    elif status == "unavailable":
        return "Under Review"
    else:
        return "Pending Review"

refunds_df = pd.DataFrame()

refunds_df["refund_id"] = [f"RF{str(i + 1).zfill(3)}" for i in range(len(refund_candidates))]
refunds_df["order_id"] = [
    get_campusgear_order_id(row["order_id"], i)
    for i, row in refund_candidates.iterrows()
]
refunds_df["original_order_id"] = refund_candidates["order_id"]
refunds_df["reason"] = refund_candidates.apply(generate_refund_reason, axis=1)
refunds_df["status"] = refund_candidates.apply(generate_refund_status, axis=1)
refunds_df["refund_amount_rm"] = refund_candidates["payment_value_rm"].fillna(0).round(2)
refunds_df["source_review_score"] = refund_candidates["review_score"].fillna("Not Available")
refunds_df["source_order_status"] = refund_candidates["order_status"].str.title()

print("CampusGear refund database created from the same Olist dataset.")
print("Final refund database shape:", refunds_df.shape)

refunds_df.head(15)

CampusGear refund database created from the same Olist dataset.
Final refund database shape: (8, 8)


,refund_id,order_id,original_order_id,reason,status,refund_amount_rm,source_review_score,source_order_status
0,RF001,CG1007,0538bda829ac14e64a201dc84fb1ba3b,Customer complaint based on low review score,Pending Review,224.69,2.0,Shipped
1,RF002,CG1008,6e77d9428b5dec0e2557b439ea6d61df,Customer complaint based on low review score,Pending Review,277.48,1.0,Shipped
2,RF003,CG1010,88743d1ed8e7b6ca01e12830855ccda2,Customer complaint based on low review score,Pending Review,103.04,1.0,Shipped
3,RF004,CG1011,5aa551497a6e4522899b4a89a3eeff5f,Order was cancelled,Completed,136.26,1.0,Canceled
4,RF005,CG1012,ab2e482aff4baa52ec6da817af85b981,Order was cancelled,Completed,186.44,1.0,Canceled
5,RF006,CG1013,881a5cda3b897b035971e3fcfce58aa6,Order was cancelled,Completed,189.37,1.0,Canceled
6,RF007,CG1014,5b0ee19a3125fc083348baa76d816ed0,Order was cancelled,Completed,90.98,1.0,Canceled
7,RF008,CG1015,f79644ec98219489ae16ee7aabac4ebb,Order was cancelled,Completed,36.32,5.0,Canceled


# Cell 8.2：Check the logic of the refund database source

In [ ]:
# Cell 8.2: Check Refund Database Source Logic

print("Refund database source logic:")
print("1. order_status = canceled     -> Order was cancelled")
print("2. order_status = unavailable  -> Product was unavailable")
print("3. review_score <= 2           -> Low review score / possible complaint")

print("\nRefund database preview:")
refunds_df[[
    "refund_id",
    "order_id",
    "reason",
    "status",
    "refund_amount_rm",
    "source_review_score",
    "source_order_status"
]].head(10)

Refund database source logic:
1. order_status = canceled     -> Order was cancelled
2. order_status = unavailable  -> Product was unavailable
3. review_score <= 2           -> Low review score / possible complaint

Refund database preview:


,refund_id,order_id,reason,status,refund_amount_rm,source_review_score,source_order_status
0,RF001,CG1007,Customer complaint based on low review score,Pending Review,224.69,2.0,Shipped
1,RF002,CG1008,Customer complaint based on low review score,Pending Review,277.48,1.0,Shipped
2,RF003,CG1010,Customer complaint based on low review score,Pending Review,103.04,1.0,Shipped
3,RF004,CG1011,Order was cancelled,Completed,136.26,1.0,Canceled
4,RF005,CG1012,Order was cancelled,Completed,186.44,1.0,Canceled
5,RF006,CG1013,Order was cancelled,Completed,189.37,1.0,Canceled
6,RF007,CG1014,Order was cancelled,Completed,90.98,1.0,Canceled
7,RF008,CG1015,Order was cancelled,Completed,36.32,5.0,Canceled


# Cell 9：Creating the CampusGear Knowledge Base

In [ ]:
# Cell 9: Create CampusGear Knowledge Base

knowledge_base = [
    """
    CampusGear is an online bookstore and stationery store. It provides books, stationery,
    office products, computer accessories, electronics, and study-related products.
    The product database is derived from the Olist e-commerce product, product category,
    and order item datasets.
    """,

    """
    CampusGear product recommendation policy: For computer science students, the chatbot should
    recommend technical books, stationery, computer accessories, electronic study tools, and study
    desk accessories. For exam revision, the chatbot should recommend books, notebooks,
    stationery products, and learning support items.
    """,

    """
    CampusGear shipping policy: Orders within Peninsular Malaysia usually take 2 to 4 working days.
    Orders to East Malaysia usually take 5 to 7 working days. Customers receive a tracking number
    after the order has been shipped.
    """,

    """
    CampusGear payment policy: Customers can pay using FPX online banking, debit card, credit card,
    and Touch 'n Go eWallet. Cash on delivery is not available.
    """,

    """
    CampusGear refund policy: Customers can request a refund within 7 days after delivery if the item is
    damaged, defective, incorrect, unavailable, cancelled, or missing. Customers should provide the order ID
    and supporting evidence such as photos for damaged items.
    """,

    """
    Refund processing usually takes 3 to 5 working days after approval. Refunds are returned to the original
    payment method used by the customer.
    """,

    """
    Order tracking policy: Customers can track their orders using the order ID. The chatbot can provide
    order status, product name, tracking number, purchase date, estimated delivery date, customer delivery date,
    payment method, payment value, and freight value.
    """
]

for i, doc in enumerate(knowledge_base):
    print(f"Document {i+1}:")
    print(doc.strip())
    print("-" * 80)

Document 1:
CampusGear is an online bookstore and stationery store. It provides books, stationery,
    office products, computer accessories, electronics, and study-related products.
    The product database is derived from the Olist e-commerce product, product category,
    and order item datasets.
--------------------------------------------------------------------------------
Document 2:
CampusGear product recommendation policy: For computer science students, the chatbot should
    recommend technical books, stationery, computer accessories, electronic study tools, and study
    desk accessories. For exam revision, the chatbot should recommend books, notebooks,
    stationery products, and learning support items.
--------------------------------------------------------------------------------
Document 3:
CampusGear shipping policy: Orders within Peninsular Malaysia usually take 2 to 4 working days.
    Orders to East Malaysia usually take 5 to 7 working days. Customers receive a tra

# Cell 10：Loading Embedding Model

In [ ]:
# Cell 10: Load Embedding Model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


# Cell 11: Converting Knowledge Bases into Embeddings

In [ ]:
# Cell 11: Convert Knowledge Base into Embeddings

kb_embeddings = embedding_model.encode(knowledge_base)

print("Knowledge base embedding shape:", kb_embeddings.shape)

Knowledge base embedding shape: (7, 384)


# Cell 12: Creating a FAISS Vector Database

In [ ]:
# Cell 12: Create FAISS Vector Database

kb_embeddings_np = np.array(kb_embeddings).astype("float32")

faiss.normalize_L2(kb_embeddings_np)

embedding_dimension = kb_embeddings_np.shape[1]

vector_index = faiss.IndexFlatIP(embedding_dimension)
vector_index.add(kb_embeddings_np)

print("FAISS vector database created successfully.")
print("Number of documents in vector database:", vector_index.ntotal)

FAISS vector database created successfully.
Number of documents in vector database: 7


# Cell 13: Creating a RAG search function

In [ ]:
# Cell 13: RAG Retrieval Function

def retrieve_context(user_query, top_k=2):
    query_embedding = embedding_model.encode([user_query])
    query_embedding_np = np.array(query_embedding).astype("float32")

    faiss.normalize_L2(query_embedding_np)

    scores, indices = vector_index.search(query_embedding_np, top_k)

    retrieved_docs = []
    for idx in indices[0]:
        retrieved_docs.append(knowledge_base[idx].strip())

    context = "\n".join(retrieved_docs)

    return context, scores[0], indices[0]

# Cell 14: Testing RAG search performance

In [ ]:
# Cell 14: Test RAG Retrieval

test_query = "What payment methods do you accept?"

context, scores, indices = retrieve_context(test_query, top_k=2)

print("User Query:")
print(test_query)

print("\nRetrieved Context:")
print(context)

print("\nSimilarity Scores:")
print(scores)

print("\nRetrieved Document Indices:")
print(indices)

User Query:
What payment methods do you accept?

Retrieved Context:
CampusGear payment policy: Customers can pay using FPX online banking, debit card, credit card,
    and Touch 'n Go eWallet. Cash on delivery is not available.
Refund processing usually takes 3 to 5 working days after approval. Refunds are returned to the original
    payment method used by the customer.

Similarity Scores:
[0.4085251  0.22505894]

Retrieved Document Indices:
[3 5]


# Cell 15：Create a Fine-Tuning dataset

In [ ]:
# Cell 15: Create Fine-Tuning Dataset

fine_tuning_data = [
    # Product recommendation
    ("Can you recommend products for a computer science student?", "product_recommendation"),
    ("What should I buy for my programming class?", "product_recommendation"),
    ("Suggest some stationery for exam revision.", "product_recommendation"),
    ("I need useful study items for university.", "product_recommendation"),
    ("What products are suitable for AI students?", "product_recommendation"),
    ("Recommend books and accessories for coding.", "product_recommendation"),
    ("Do you have study products for computer science?", "product_recommendation"),
    ("Which items are good for lecture note taking?", "product_recommendation"),

    # Shipping policy
    ("How long does shipping take?", "shipping_policy"),
    ("What is your delivery time?", "shipping_policy"),
    ("How many days to deliver to East Malaysia?", "shipping_policy"),
    ("How fast can I receive my order?", "shipping_policy"),
    ("Do you ship to Peninsular Malaysia?", "shipping_policy"),
    ("When will my order arrive?", "shipping_policy"),
    ("Tell me about your shipping policy.", "shipping_policy"),
    ("How long is delivery to Sabah or Sarawak?", "shipping_policy"),

    # Payment method
    ("What payment methods do you accept?", "payment_method"),
    ("Can I pay using eWallet?", "payment_method"),
    ("Do you accept credit card?", "payment_method"),
    ("Can I use FPX online banking?", "payment_method"),
    ("Is cash on delivery available?", "payment_method"),
    ("Can I pay with debit card?", "payment_method"),
    ("Tell me your payment options.", "payment_method"),
    ("Do you accept Touch n Go eWallet?", "payment_method"),

    # Order tracking
    ("Track my order CG1001.", "order_tracking"),
    ("Where is my order CG1002?", "order_tracking"),
    ("Can you check order CG1003?", "order_tracking"),
    ("I want to know the status of CG1004.", "order_tracking"),
    ("Please track my order.", "order_tracking"),
    ("Check delivery status for CG1001.", "order_tracking"),
    ("Has my order been shipped?", "order_tracking"),
    ("What is the tracking number for CG1003?", "order_tracking"),

    # Refund processing
    ("I want to refund my order.", "refund_processing"),
    ("My book is damaged. Can I get a refund?", "refund_processing"),
    ("I received the wrong item.", "refund_processing"),
    ("How can I request a refund?", "refund_processing"),
    ("Please process refund for CG1003.", "refund_processing"),
    ("I want to return a defective product.", "refund_processing"),
    ("My item is missing from the package.", "refund_processing"),
    ("Can I get my money back?", "refund_processing"),

    # General FAQ
    ("Hello", "general_faq"),
    ("Hi chatbot", "general_faq"),
    ("Who are you?", "general_faq"),
    ("What can you do?", "general_faq"),
    ("Help me", "general_faq"),
    ("What is CampusGear?", "general_faq"),
    ("Good morning", "general_faq"),
    ("Thank you", "general_faq")
]

fine_tuning_df = pd.DataFrame(fine_tuning_data, columns=["text", "intent"])

fine_tuning_df.head(10)

,text,intent
0,Can you recommend products for a computer scie...,product_recommendation
1,What should I buy for my programming class?,product_recommendation
2,Suggest some stationery for exam revision.,product_recommendation
3,I need useful study items for university.,product_recommendation
4,What products are suitable for AI students?,product_recommendation
5,Recommend books and accessories for coding.,product_recommendation
6,Do you have study products for computer science?,product_recommendation
7,Which items are good for lecture note taking?,product_recommendation
8,How long does shipping take?,shipping_policy
9,What is your delivery time?,shipping_policy


# Cell 16：Check the number of each Intent

In [ ]:
# Cell 16: Check Intent Class Distribution

fine_tuning_df["intent"].value_counts()

,count
intent,
product_recommendation,8
shipping_policy,8
payment_method,8
order_tracking,8
refund_processing,8
general_faq,8


# Cell **17：Tag Encoding**

In [ ]:
# Cell 17: Label Encoding

label_encoder = LabelEncoder()

fine_tuning_df["label"] = label_encoder.fit_transform(fine_tuning_df["intent"])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print("Label Mapping:")
label_mapping

Label Mapping:


{'general_faq': np.int64(0),
 'order_tracking': np.int64(1),
 'payment_method': np.int64(2),
 'product_recommendation': np.int64(3),
 'refund_processing': np.int64(4),
 'shipping_policy': np.int64(5)}

# Cell 18：Divide into training and test sets

In [ ]:
# Cell 18: Split Dataset into Training and Testing Sets

train_texts, test_texts, train_labels, test_labels = train_test_split(
    fine_tuning_df["text"].tolist(),
    fine_tuning_df["label"].tolist(),
    test_size=0.25,
    random_state=SEED,
    stratify=fine_tuning_df["label"]
)

print("Number of training samples:", len(train_texts))
print("Number of testing samples:", len(test_texts))

Number of training samples: 36
Number of testing samples: 12


# Cell 19：Loading pre-trained models

In [ ]:
# Cell 19: Load Pre-Trained Model for Fine-Tuning

intent_model_name = "distilbert-base-uncased"

intent_tokenizer = AutoTokenizer.from_pretrained(intent_model_name)

num_labels = len(label_encoder.classes_)

intent_model = AutoModelForSequenceClassification.from_pretrained(
    intent_model_name,
    num_labels=num_labels
)

intent_model.to(device)

print("Intent classification model loaded successfully.")
print("Model name:", intent_model_name)
print("Number of labels:", num_labels)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Intent classification model loaded successfully.
Model name: distilbert-base-uncased
Number of labels: 6


# Cell 20：Creating PyTorch Datasets and DataLoaders

In [ ]:
# Cell 20: Create PyTorch Dataset and DataLoader

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

        return item


train_dataset = IntentDataset(train_texts, train_labels, intent_tokenizer)
test_dataset = IntentDataset(test_texts, test_labels, intent_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print("Dataset and DataLoader created successfully.")

Dataset and DataLoader created successfully.


# Cell 21: Fine-tuning the Intent Classification Model

In [ ]:
# Cell 21: Fine-Tune Intent Classification Model

optimizer = AdamW(intent_model.parameters(), lr=2e-5)

epochs = 5

training_history = []

for epoch in range(epochs):
    intent_model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = intent_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    training_history.append(average_loss)

    print(f"Epoch {epoch + 1}/{epochs} - Training Loss: {average_loss:.4f}")

Epoch 1/5 - Training Loss: 1.8126
Epoch 2/5 - Training Loss: 1.7653
Epoch 3/5 - Training Loss: 1.7007
Epoch 4/5 - Training Loss: 1.6198
Epoch 5/5 - Training Loss: 1.4918


# Cell 22: Testing the Fine-Tuned Model

In [ ]:
# Cell 22: Evaluate Fine-Tuned Intent Classification Model

intent_model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = intent_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_true_labels, all_predictions)

print("Intent Classification Accuracy:", round(accuracy, 4))
print("\nClassification Report:")
print(
    classification_report(
        all_true_labels,
        all_predictions,
        target_names=label_encoder.classes_
    )
)

Intent Classification Accuracy: 0.5

Classification Report:
                        precision    recall  f1-score   support

           general_faq       1.00      0.50      0.67         2
        order_tracking       0.50      0.50      0.50         2
        payment_method       0.33      1.00      0.50         2
product_recommendation       0.67      1.00      0.80         2
     refund_processing       0.00      0.00      0.00         2
       shipping_policy       0.00      0.00      0.00         2

              accuracy                           0.50        12
             macro avg       0.42      0.50      0.41        12
          weighted avg       0.42      0.50      0.41        12



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Cell 23：Save Fine-Tuned Model

In [ ]:
# Cell 23: Save Fine-Tuned Intent Model

intent_model.save_pretrained("campusgear_finetuned_intent_model")
intent_tokenizer.save_pretrained("campusgear_finetuned_intent_model")

print("Fine-tuned intent model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned intent model saved successfully.


# Cell 24：Create an Intent Prediction function

In [ ]:
# Cell 24: Intent Prediction Function

def predict_intent(user_query):
    intent_model.eval()

    encoded = intent_tokenizer(
        user_query,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = intent_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    predicted_label = torch.argmax(outputs.logits, dim=1).item()
    predicted_intent = label_encoder.inverse_transform([predicted_label])[0]

    return predicted_intent

# Cell 24.1：Hybrid Intent Detection

In [ ]:
# Cell 24.1: Hybrid Intent Detection Function

def detect_intent_hybrid(user_query):
    query_lower = user_query.lower()

    # Refund policy question should use RAG
    if "refund policy" in query_lower or "how can i request a refund" in query_lower:
        return "refund_policy"

    # Rule-based safety routing for refund processing
    if (
        "refund" in query_lower
        or "return" in query_lower
        or "damaged" in query_lower
        or "wrong item" in query_lower
        or "defective" in query_lower
        or "missing" in query_lower
    ):
        return "refund_processing"

    # Order tracking
    if (
        "track" in query_lower
        or "order status" in query_lower
        or "where is my order" in query_lower
        or re.search(r"CG\d{4}", user_query.upper())
    ):
        return "order_tracking"

    # Payment
    if (
        "payment" in query_lower
        or "pay" in query_lower
        or "ewallet" in query_lower
        or "credit card" in query_lower
        or "debit card" in query_lower
        or "fpx" in query_lower
    ):
        return "payment_method"

    # Shipping
    if (
        "shipping" in query_lower
        or "delivery" in query_lower
        or "deliver" in query_lower
        or "east malaysia" in query_lower
        or "peninsular malaysia" in query_lower
    ):
        return "shipping_policy"

    # Product recommendation
    if (
        "recommend" in query_lower
        or "suggest" in query_lower
        or "buy" in query_lower
        or "product" in query_lower
        or "book" in query_lower
        or "stationery" in query_lower
        or "study" in query_lower
        or "programming" in query_lower
        or "coding" in query_lower
    ):
        return "product_recommendation"

    # If no rule is matched, use fine-tuned model
    return predict_intent(user_query)

# Cell 25：Test Hybrid Intent Detection

In [ ]:
# Cell 25: Test Hybrid Intent Prediction

sample_queries = [
    "Can you recommend something for my programming class?",
    "How long does delivery take?",
    "Can I pay using credit card?",
    "Track my order CG1001.",
    "My book is damaged and I want a refund.",
    "What is your refund policy for damaged items?",
    "Hello, what can you do?"
]

for query in sample_queries:
    print("User Query:", query)
    print("Predicted Intent:", detect_intent_hybrid(query))
    print("-" * 80)

User Query: Can you recommend something for my programming class?
Predicted Intent: product_recommendation
--------------------------------------------------------------------------------
User Query: How long does delivery take?
Predicted Intent: shipping_policy
--------------------------------------------------------------------------------
User Query: Can I pay using credit card?
Predicted Intent: payment_method
--------------------------------------------------------------------------------
User Query: Track my order CG1001.
Predicted Intent: order_tracking
--------------------------------------------------------------------------------
User Query: My book is damaged and I want a refund.
Predicted Intent: refund_processing
--------------------------------------------------------------------------------
User Query: What is your refund policy for damaged items?
Predicted Intent: refund_policy
--------------------------------------------------------------------------------
User Query: 

# Cell 26: Loading LLM Backend

In [ ]:
# Cell 26: Load LLM Backend Model

llm_model_name = "google/flan-t5-small"

llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

llm_model.to(device)

print("LLM backend loaded successfully:", llm_model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM backend loaded successfully: google/flan-t5-small


# Cell 27: Creating an LLM response generation function

In [ ]:
# Cell 27: LLM Response Generation Function

def generate_llm_response(user_query, context):
    prompt = f"""
You are a helpful customer service chatbot for CampusGear Online Bookstore and Stationery Store.

Use the following company information to answer the customer question.

Company Information:
{context}

Customer Question:
{user_query}

Answer in a clear and helpful way:
"""

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True
    )

    response = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if len(response.strip()) < 10:
        response = context

    return response

# Cell 28：Extract Order ID

In [ ]:
# Cell 28: Extract Order ID from User Query

def extract_order_id(user_query):
    match = re.search(r"CG\d{4}", user_query.upper())

    if match:
        return match.group(0)
    else:
        return None

# Cell 29：Product recommendation function

In [ ]:
# Cell 29: Product Recommendation Function Based on Unified Olist Product Dataset

def recommend_products(user_query):
    query_lower = user_query.lower()

    if (
        "computer science" in query_lower
        or "programming" in query_lower
        or "coding" in query_lower
        or "ai" in query_lower
        or "algorithm" in query_lower
    ):
        recommended_products = products_df[
            products_df["category"].str.contains(
                "Book|Computer|Electronic|Stationery",
                case=False,
                na=False
            )
        ].head(8)

    elif (
        "exam" in query_lower
        or "revision" in query_lower
        or "note" in query_lower
        or "study" in query_lower
    ):
        recommended_products = products_df[
            products_df["category"].str.contains(
                "Book|Stationery",
                case=False,
                na=False
            )
        ].head(8)

    else:
        recommended_products = products_df.head(8)

    response = "Here are some recommended products from CampusGear:\n"

    for _, row in recommended_products.iterrows():
        response += (
            f"- {row['product_name']} "
            f"({row['category']}): RM{row['price_rm']:.2f}. "
            f"{row['description']}\n"
        )

    return response

# Cell 30: Order Query Function

In [ ]:
# Cell 30: Order Tracking Function Based on Unified Olist Dataset

def track_order(user_query):
    order_id = extract_order_id(user_query)

    if order_id is None:
        return "Please provide a valid order ID, such as CG1001, so I can track your order."

    result = orders_df[orders_df["order_id"] == order_id]

    if result.empty:
        return f"Sorry, order {order_id} was not found in the CampusGear order database."

    row = result.iloc[0]

    response = (
        f"Order {row['order_id']} is currently {row['status']}.\n"
        f"Product: {row['product_name']}.\n"
        f"Tracking number: {row['tracking_no']}.\n"
        f"Purchase date: {row['purchase_date']}.\n"
        f"Estimated delivery date: {row['estimated_delivery_date']}.\n"
        f"Customer delivery date: {row['customer_delivery_date']}.\n"
        f"Payment method: {row['payment_method']}.\n"
        f"Payment value: RM{row['payment_value_rm']:.2f}.\n"
        f"Freight value: RM{row['freight_value_rm']:.2f}."
    )

    return response

# Cell 31: Refund Processing Function


In [ ]:
# Cell 31: Refund Processing Function Based on Unified Olist Dataset

def process_refund(user_query):
    order_id = extract_order_id(user_query)

    if order_id is None:
        return (
            "To process a refund, please provide your order ID, such as CG1001. "
            "For damaged items, please also provide supporting evidence such as photos."
        )

    existing_refund = refunds_df[refunds_df["order_id"] == order_id]

    if not existing_refund.empty:
        refund_row = existing_refund.iloc[0]

        return (
            f"A refund record already exists for order {order_id}.\n"
            f"Refund ID: {refund_row['refund_id']}.\n"
            f"Reason: {refund_row['reason']}.\n"
            f"Refund status: {refund_row['status']}.\n"
            f"Refund amount: RM{refund_row['refund_amount_rm']:.2f}.\n"
            f"Source order status from dataset: {refund_row['source_order_status']}.\n"
            f"Source review score from dataset: {refund_row['source_review_score']}."
        )

    order_result = orders_df[orders_df["order_id"] == order_id]

    if order_result.empty:
        return f"Refund request failed because order {order_id} was not found."

    reason = "Customer refund request"

    if "damaged" in user_query.lower():
        reason = "Damaged item"
    elif "wrong" in user_query.lower():
        reason = "Wrong item received"
    elif "missing" in user_query.lower():
        reason = "Missing item"
    elif "defective" in user_query.lower():
        reason = "Defective item"
    elif "cancel" in user_query.lower():
        reason = "Order cancellation"

    row = order_result.iloc[0]

    response = (
        f"Refund request for order {order_id} has been submitted.\n"
        f"Product: {row['product_name']}.\n"
        f"Reason: {reason}.\n"
        f"Order status: {row['status']}.\n"
        f"Estimated refund amount: RM{row['payment_value_rm']:.2f}.\n"
        f"The request will be reviewed within 3 to 5 working days.\n"
        f"If the item is damaged, please provide photo evidence."
    )

    return response

# Cell 32：General FAQ Reply Function

In [ ]:
# Cell 32: General FAQ Response Function

def general_faq_response():
    return (
        "Hello! I am the CampusGear customer service chatbot. "
        "I can help you with product suggestions, shipping policies, payment methods, "
        "order tracking, and refund processing."
    )

# Cell 33：Main Chatbot function

In [ ]:
# Cell 33: Main CampusGear Chatbot Function

def campusgear_chatbot(user_query):
    predicted_intent = detect_intent_hybrid(user_query)

    if predicted_intent == "product_recommendation":
        response = recommend_products(user_query)

    elif predicted_intent == "order_tracking":
        response = track_order(user_query)

    elif predicted_intent == "refund_processing":
        response = process_refund(user_query)

    elif predicted_intent in ["shipping_policy", "payment_method", "refund_policy"]:
        context, scores, indices = retrieve_context(user_query, top_k=2)
        response = generate_llm_response(user_query, context)

    elif predicted_intent == "general_faq":
        response = general_faq_response()

    else:
        context, scores, indices = retrieve_context(user_query, top_k=2)
        response = generate_llm_response(user_query, context)

    return {
        "user_query": user_query,
        "predicted_intent": predicted_intent,
        "chatbot_response": response
    }

# Cell 34：Single question test

In [ ]:
# Cell 34: Single Query Test

result = campusgear_chatbot("What payment methods do you accept?")

print("User Query:")
print(result["user_query"])

print("\nPredicted Intent:")
print(result["predicted_intent"])

print("\nChatbot Response:")
print(result["chatbot_response"])

User Query:
What payment methods do you accept?

Predicted Intent:
payment_method

Chatbot Response:
FPX online banking, debit card, credit card, and Touch 'n Go eWallet


# Cell 35：Prepare to test order number

In [ ]:
# Cell 35: Prepare Dynamic Test Order IDs

valid_order_id_1 = orders_df.iloc[0]["order_id"]
valid_order_id_2 = orders_df.iloc[1]["order_id"]

if len(refunds_df) > 0:
    refund_order_id = refunds_df.iloc[0]["order_id"]
else:
    refund_order_id = valid_order_id_1

invalid_order_id = "CG9999"

print("Valid order ID 1:", valid_order_id_1)
print("Valid order ID 2:", valid_order_id_2)
print("Refund order ID:", refund_order_id)
print("Invalid order ID:", invalid_order_id)

Valid order ID 1: CG1001
Valid order ID 2: CG1002
Refund order ID: CG1007
Invalid order ID: CG9999


# Cell 36：Preparing to simulate customer input

In [ ]:
# Cell 36: Simulated Customer Input Queries

test_queries = [
    "I am a computer science student. Can you recommend useful study products?",
    "How long does shipping take to Peninsular Malaysia?",
    "What payment methods do you accept?",
    "Can I pay using Touch n Go eWallet?",
    f"Track my order {valid_order_id_1}.",
    f"Where is my order {valid_order_id_2}?",
    f"Track order {invalid_order_id}.",
    f"I received a damaged item. I want to refund order {refund_order_id}.",
    f"I received the wrong item. Please process refund for {valid_order_id_1}.",
    "What is your refund policy for damaged items?",
    "Can I pay with eWallet and receive my order in East Malaysia?",
    "Hello, what can you do?"
]

for query in test_queries:
    print(query)

I am a computer science student. Can you recommend useful study products?
How long does shipping take to Peninsular Malaysia?
What payment methods do you accept?
Can I pay using Touch n Go eWallet?
Track my order CG1001.
Where is my order CG1002?
Track order CG9999.
I received a damaged item. I want to refund order CG1007.
I received the wrong item. Please process refund for CG1001.
What is your refund policy for damaged items?
Can I pay with eWallet and receive my order in East Malaysia?
Hello, what can you do?


# Cell 37：Run a full Chatbot test

In [ ]:
# Cell 37: Run Complete Chatbot Simulation

chatbot_results = []

for query in test_queries:
    result = campusgear_chatbot(query)
    chatbot_results.append(result)

    print("Customer:", result["user_query"])
    print("Predicted Intent:", result["predicted_intent"])
    print("Chatbot:", result["chatbot_response"])
    print("=" * 100)

Customer: I am a computer science student. Can you recommend useful study products?
Predicted Intent: product_recommendation
Chatbot: Here are some recommended products from CampusGear:
- CampusGear Computer Accessory 1 (Computers Accessories): RM137.65. A CampusGear product from the Computers Accessories category. This product is suitable for students, academic study, office work, or learning support. The product information is derived from the Olist e-commerce product and order item dataset.
- CampusGear Computer Accessory 2 (Computers Accessories): RM149.94. A CampusGear product from the Computers Accessories category. This product is suitable for students, academic study, office work, or learning support. The product information is derived from the Olist e-commerce product and order item dataset.
- CampusGear Computer Accessory 3 (Computers Accessories): RM84.37. A CampusGear product from the Computers Accessories category. This product is suitable for students, academic study, off

# Cell 38：Convert the test results into a table

In [ ]:
# Cell 38: Convert Chatbot Results into DataFrame

chatbot_results_df = pd.DataFrame(chatbot_results)

chatbot_results_df

,user_query,predicted_intent,chatbot_response
0,I am a computer science student. Can you recom...,product_recommendation,Here are some recommended products from Campus...
1,How long does shipping take to Peninsular Mala...,shipping_policy,2 to 4 working days
2,What payment methods do you accept?,payment_method,"FPX online banking, debit card, credit card, a..."
3,Can I pay using Touch n Go eWallet?,payment_method,CampusGear payment policy: Customers can pay u...
4,Track my order CG1001.,order_tracking,Order CG1001 is currently Delivered.\nProduct:...
5,Where is my order CG1002?,order_tracking,Order CG1002 is currently Delivered.\nProduct:...
6,Track order CG9999.,order_tracking,"Sorry, order CG9999 was not found in the Campu..."
7,I received a damaged item. I want to refund or...,refund_processing,A refund record already exists for order CG100...
8,I received the wrong item. Please process refu...,refund_processing,Refund request for order CG1001 has been submi...
9,What is your refund policy for damaged items?,refund_policy,Customer service chatbot


# Cell 39：Create Evaluation Dataset

In [ ]:
# Cell 39: Create Evaluation Dataset

evaluation_data = [
    ("I am a computer science student. Can you recommend useful study products?", "product_recommendation"),
    ("How long does shipping take to Peninsular Malaysia?", "shipping_policy"),
    ("What payment methods do you accept?", "payment_method"),
    ("Can I pay using Touch n Go eWallet?", "payment_method"),
    (f"Track my order {valid_order_id_1}.", "order_tracking"),
    (f"Where is my order {valid_order_id_2}?", "order_tracking"),
    (f"Track order {invalid_order_id}.", "order_tracking"),
    (f"I received a damaged item. I want to refund order {refund_order_id}.", "refund_processing"),
    (f"I received the wrong item. Please process refund for {valid_order_id_1}.", "refund_processing"),
    ("What is your refund policy for damaged items?", "refund_policy"),
    ("Can I pay with eWallet and receive my order in East Malaysia?", "payment_method"),
    ("Hello, what can you do?", "general_faq")
]

evaluation_df = pd.DataFrame(evaluation_data, columns=["query", "expected_intent"])

evaluation_df

,query,expected_intent
0,I am a computer science student. Can you recom...,product_recommendation
1,How long does shipping take to Peninsular Mala...,shipping_policy
2,What payment methods do you accept?,payment_method
3,Can I pay using Touch n Go eWallet?,payment_method
4,Track my order CG1001.,order_tracking
5,Where is my order CG1002?,order_tracking
6,Track order CG9999.,order_tracking
7,I received a damaged item. I want to refund or...,refund_processing
8,I received the wrong item. Please process refu...,refund_processing
9,What is your refund policy for damaged items?,refund_policy


# Cell 40：Calculate Intent Accuracy

In [ ]:
# Cell 40: Calculate Intent Classification Accuracy

predicted_intents = []

for query in evaluation_df["query"]:
    predicted_intents.append(detect_intent_hybrid(query))

evaluation_df["predicted_intent"] = predicted_intents

evaluation_df["intent_correct"] = evaluation_df["expected_intent"] == evaluation_df["predicted_intent"]

intent_accuracy = evaluation_df["intent_correct"].mean()

print("Intent Classification Accuracy:", round(intent_accuracy * 100, 2), "%")

evaluation_df

Intent Classification Accuracy: 91.67 %


,query,expected_intent,predicted_intent,intent_correct
0,I am a computer science student. Can you recom...,product_recommendation,product_recommendation,True
1,How long does shipping take to Peninsular Mala...,shipping_policy,shipping_policy,True
2,What payment methods do you accept?,payment_method,payment_method,True
3,Can I pay using Touch n Go eWallet?,payment_method,payment_method,True
4,Track my order CG1001.,order_tracking,order_tracking,True
5,Where is my order CG1002?,order_tracking,order_tracking,True
6,Track order CG9999.,order_tracking,order_tracking,True
7,I received a damaged item. I want to refund or...,refund_processing,refund_processing,True
8,I received the wrong item. Please process refu...,refund_processing,refund_processing,True
9,What is your refund policy for damaged items?,refund_policy,refund_policy,True


# Cell 41：Calculate Task Completion Rate

In [ ]:
# Cell 41: Calculate Task Completion Rate

task_completion_results = []

for query in evaluation_df["query"]:
    result = campusgear_chatbot(query)
    response = result["chatbot_response"].lower()

    completed = False
    query_lower = query.lower()

    if "recommend" in query_lower or "student" in query_lower:
        completed = "recommended products" in response or "campusgear" in response or "book" in response

    elif "shipping" in query_lower or "delivery" in query_lower or "east malaysia" in query_lower:
        completed = "working days" in response or "delivery" in response or "shipping" in response

    elif "payment" in query_lower or "pay" in query_lower or "ewallet" in query_lower:
        completed = "fpx" in response or "credit" in response or "ewallet" in response or "debit" in response

    elif "track" in query_lower or "where is my order" in query_lower:
        completed = "order" in response and ("currently" in response or "not found" in response or "status" in response)

    elif "refund policy" in query_lower:
        completed = "refund" in response and ("7 days" in response or "damaged" in response or "approval" in response)

    elif "refund" in query_lower or "damaged" in query_lower or "wrong item" in query_lower:
        completed = "refund" in response or "reviewed" in response or "refund record" in response

    elif "hello" in query_lower:
        completed = "help" in response or "chatbot" in response

    task_completion_results.append(completed)

evaluation_df["task_completed"] = task_completion_results

task_completion_rate = evaluation_df["task_completed"].mean()

print("Task Completion Rate:", round(task_completion_rate * 100, 2), "%")

evaluation_df

Task Completion Rate: 91.67 %


,query,expected_intent,predicted_intent,intent_correct,task_completed
0,I am a computer science student. Can you recom...,product_recommendation,product_recommendation,True,True
1,How long does shipping take to Peninsular Mala...,shipping_policy,shipping_policy,True,True
2,What payment methods do you accept?,payment_method,payment_method,True,True
3,Can I pay using Touch n Go eWallet?,payment_method,payment_method,True,True
4,Track my order CG1001.,order_tracking,order_tracking,True,True
5,Where is my order CG1002?,order_tracking,order_tracking,True,True
6,Track order CG9999.,order_tracking,order_tracking,True,True
7,I received a damaged item. I want to refund or...,refund_processing,refund_processing,True,True
8,I received the wrong item. Please process refu...,refund_processing,refund_processing,True,True
9,What is your refund policy for damaged items?,refund_policy,refund_policy,True,False


# Cell 42：Statistical results by task type

In [ ]:
# Cell 42: Evaluation Summary by Task Type

summary_table = evaluation_df.groupby("expected_intent").agg(
    total_queries=("query", "count"),
    correct_intent=("intent_correct", "sum"),
    completed_tasks=("task_completed", "sum")
).reset_index()

summary_table["intent_accuracy_percent"] = (
    summary_table["correct_intent"] / summary_table["total_queries"] * 100
).round(2)

summary_table["task_completion_percent"] = (
    summary_table["completed_tasks"] / summary_table["total_queries"] * 100
).round(2)

summary_table

,expected_intent,total_queries,correct_intent,completed_tasks,intent_accuracy_percent,task_completion_percent
0,general_faq,1,0,1,0.0,100.0
1,order_tracking,3,3,3,100.0,100.0
2,payment_method,3,3,3,100.0,100.0
3,product_recommendation,1,1,1,100.0,100.0
4,refund_policy,1,1,0,100.0,0.0
5,refund_processing,2,2,2,100.0,100.0
6,shipping_policy,1,1,1,100.0,100.0


# Cell 43：Save the results file

In [ ]:
# Cell 43: Save Chatbot Output, Evaluation Results, and Databases

chatbot_results_df.to_csv("campusgear_chatbot_simulated_outputs.csv", index=False)
evaluation_df.to_csv("campusgear_chatbot_evaluation_results.csv", index=False)
summary_table.to_csv("campusgear_chatbot_evaluation_summary.csv", index=False)

products_df.to_csv("campusgear_product_database_from_olist.csv", index=False)
orders_df.to_csv("campusgear_order_database_from_olist.csv", index=False)
refunds_df.to_csv("campusgear_refund_database_from_olist.csv", index=False)

print("All result files saved successfully.")

All result files saved successfully.


# Cell 44：Interactive testing

In [ ]:
# ============================================================
# Cell 44: CampusGear Futuristic Tech-Style Gradio Interface
# Modern App Chat + Auto Scroll Version
# ============================================================

!pip install -q gradio

import html
import gradio as gr
import pandas as pd


# ============================================================
# 1. Prepare Dynamic Example Questions
# ============================================================

def prepare_order_examples(order_database=None, refund_database=None):
    if order_database is None or len(order_database) == 0:
        return {
            "available_order_ids": ["CG1001", "CG1002", "CG1003", "CG1004"],
            "track_order_id": "CG1001",
            "refund_order_id": "CG1003",
            "not_found_order_id": "CG9999"
        }

    available_order_ids = order_database["order_id"].astype(str).tolist()
    track_order_id = available_order_ids[0]

    if refund_database is not None and len(refund_database) > 0:
        refund_order_id = str(refund_database.iloc[0]["order_id"])
    else:
        refund_order_id = available_order_ids[0]

    return {
        "available_order_ids": available_order_ids,
        "track_order_id": track_order_id,
        "refund_order_id": refund_order_id,
        "not_found_order_id": "CG9999"
    }


if "orders_df" in globals():
    if "refunds_df" in globals():
        order_examples = prepare_order_examples(orders_df, refunds_df)
    else:
        order_examples = prepare_order_examples(orders_df, None)
else:
    order_examples = prepare_order_examples(None, None)


available_order_ids = order_examples["available_order_ids"]
track_order_id = order_examples["track_order_id"]
refund_order_id = order_examples["refund_order_id"]
not_found_order_id = order_examples["not_found_order_id"]


example_questions = [
    "I am a computer science student. Can you recommend useful study products?",
    "Can you recommend books and stationery for exam revision?",
    "What products are suitable for coding and programming study?",
    "Suggest some useful CampusGear products for university students.",
    "What payment methods do you accept?",
    "Can I pay using online banking?",
    "Do you support e-wallet payment?",
    "How long does shipping take to Peninsular Malaysia?",
    "How long does shipping take to East Malaysia?",
    "Do you provide tracking number after shipping?",
    f"Can you track my order {track_order_id}?",
    f"What is the status of order {track_order_id}?",
    f"Track order {not_found_order_id}.",
    f"I received a damaged item. I want to refund order {refund_order_id}.",
    f"I received the wrong item. Please process refund for {track_order_id}.",
    "What is your refund policy for damaged items?"
]

order_tags_html = " ".join(
    [f'<span class="order-tag">{oid}</span>' for oid in available_order_ids[:8]]
)


# ============================================================
# 2. CSS Style
# ============================================================

custom_css = """
body, .gradio-container, .app, .main, .wrap, #root {
    background:
        radial-gradient(circle at top left, rgba(34, 211, 238, 0.25), transparent 30%),
        radial-gradient(circle at top right, rgba(124, 58, 237, 0.30), transparent 30%),
        radial-gradient(circle at bottom left, rgba(37, 99, 235, 0.24), transparent 32%),
        radial-gradient(circle at bottom right, rgba(14, 165, 233, 0.20), transparent 32%),
        linear-gradient(135deg, #020617 0%, #07111f 42%, #0f172a 72%, #111827 100%) !important;
}

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
    font-family: 'Inter', 'Segoe UI', Arial, sans-serif !important;
}

/* Main shell */
#app-shell {
    padding: 22px;
    border-radius: 28px;
    background:
        linear-gradient(
            135deg,
            rgba(15, 23, 42, 0.92),
            rgba(30, 41, 59, 0.86),
            rgba(30, 27, 75, 0.88)
        ) !important;
    border: 1px solid rgba(125, 211, 252, 0.35);
    box-shadow:
        0 0 36px rgba(14, 165, 233, 0.18),
        0 18px 42px rgba(0, 0, 0, 0.32),
        inset 0 0 28px rgba(255, 255, 255, 0.03);
}

/* Header */
#main-title {
    text-align: center;
    padding: 32px 26px;
    border-radius: 26px;
    background:
        radial-gradient(circle at top right, rgba(34, 211, 238, 0.32), transparent 36%),
        radial-gradient(circle at bottom left, rgba(168, 85, 247, 0.28), transparent 36%),
        linear-gradient(135deg, rgba(8, 47, 73, 0.98), rgba(30, 64, 175, 0.95), rgba(88, 28, 135, 0.92)) !important;
    color: white;
    margin-bottom: 22px;
    border: 1px solid rgba(125, 211, 252, 0.48);
    box-shadow:
        0 0 34px rgba(34, 211, 238, 0.20),
        0 0 34px rgba(124, 58, 237, 0.16);
}

#main-title h1 {
    font-size: 36px;
    font-weight: 900;
    margin-bottom: 10px;
    letter-spacing: 0.3px;
    color: #f8fafc !important;
    text-shadow: 0 0 18px rgba(125, 211, 252, 0.42);
}

#main-title p {
    font-size: 16px;
    color: #dbeafe !important;
    margin: 0;
    font-weight: 700;
}

/* Badges */
.ai-badge {
    display: inline-block;
    padding: 8px 15px;
    margin: 8px 5px 0 5px;
    border-radius: 999px;
    color: #e0f2fe !important;
    background: rgba(15, 23, 42, 0.62) !important;
    border: 1px solid rgba(125, 211, 252, 0.42);
    font-size: 13px;
    font-weight: 900;
    box-shadow: 0 0 14px rgba(14, 165, 233, 0.12);
}

/* Section title */
.section-title {
    margin: 24px 0 12px 0;
    padding-left: 13px;
    border-left: 5px solid #22d3ee;
    color: #e0f2fe !important;
    font-size: 18px;
    font-weight: 900;
}

/* How-to box */
.howto-box {
    padding: 18px;
    border-radius: 18px;
    background:
        linear-gradient(
            145deg,
            rgba(15, 23, 42, 0.90),
            rgba(30, 41, 59, 0.82),
            rgba(8, 47, 73, 0.72)
        ) !important;
    border: 1px solid rgba(34, 211, 238, 0.34);
    color: #dbeafe !important;
    line-height: 1.72;
    margin-bottom: 16px;
    box-shadow:
        0 0 22px rgba(14, 165, 233, 0.12),
        inset 0 0 20px rgba(255, 255, 255, 0.025);
    font-size: 15px;
    font-weight: 700;
}

.howto-box b {
    color: #67e8f9 !important;
}

.howto-box code {
    background: rgba(2, 6, 23, 0.78) !important;
    color: #a5f3fc !important;
    padding: 4px 8px;
    border-radius: 8px;
    border: 1px solid rgba(125, 211, 252, 0.32);
}

/* Order tags */
.order-tag {
    display: inline-block;
    padding: 6px 11px;
    margin: 4px;
    border-radius: 999px;
    background: rgba(37, 99, 235, 0.26) !important;
    border: 1px solid rgba(125, 211, 252, 0.35);
    color: #bfdbfe !important;
    font-size: 13px;
    font-weight: 900;
}

/* Modern App Chat Card */
.chat-app-card {
    border-radius: 28px;
    overflow: hidden;
    border: 1px solid rgba(125, 211, 252, 0.38);
    background: rgba(15, 23, 42, 0.86) !important;
    box-shadow:
        0 20px 42px rgba(0, 0, 0, 0.40),
        0 0 28px rgba(14, 165, 233, 0.16);
}

.chat-app-header {
    padding: 16px 20px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    background:
        linear-gradient(135deg, rgba(14, 165, 233, 0.38), rgba(37, 99, 235, 0.40), rgba(124, 58, 237, 0.38)) !important;
    border-bottom: 1px solid rgba(125, 211, 252, 0.22);
}

.chat-app-title {
    font-size: 18px;
    font-weight: 900;
    color: #f8fafc !important;
    margin-bottom: 4px;
}

.chat-app-subtitle {
    font-size: 13px;
    color: #bfdbfe !important;
    font-weight: 750;
}

.chat-app-status {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 8px 12px;
    border-radius: 999px;
    background: rgba(15, 23, 42, 0.58) !important;
    border: 1px solid rgba(74, 222, 128, 0.38);
    color: #bbf7d0 !important;
    font-size: 13px;
    font-weight: 900;
}

.chat-status-dot {
    width: 9px;
    height: 9px;
    border-radius: 50%;
    background: #22c55e;
    box-shadow: 0 0 12px #22c55e;
}

/* Chat window */
/* Keep auto-scroll effect */
.chat-display {
    height: 470px;
    max-height: 470px;
    overflow-y: auto;
    padding: 22px 18px;
    background:
        linear-gradient(rgba(15, 23, 42, 0.94), rgba(15, 23, 42, 0.94)),
        repeating-linear-gradient(
            0deg,
            rgba(125, 211, 252, 0.06) 0px,
            rgba(125, 211, 252, 0.06) 1px,
            transparent 1px,
            transparent 32px
        ),
        repeating-linear-gradient(
            90deg,
            rgba(125, 211, 252, 0.045) 0px,
            rgba(125, 211, 252, 0.045) 1px,
            transparent 1px,
            transparent 32px
        ) !important;
    display: flex;
    flex-direction: column-reverse;
    gap: 14px;
    scroll-behavior: smooth;
}

.empty-chat {
    height: 100%;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #94a3b8 !important;
    font-size: 16px;
    font-weight: 800;
}

/* Chat row */
.message-row {
    display: flex;
    align-items: flex-end;
    gap: 10px;
}

.user-row {
    justify-content: flex-end;
}

.assistant-row {
    justify-content: flex-start;
}

.chat-avatar {
    width: 36px;
    height: 36px;
    border-radius: 50%;
    flex-shrink: 0;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 12px;
    font-weight: 900;
    box-shadow: 0 4px 12px rgba(0,0,0,0.24);
}

.user-avatar {
    background: linear-gradient(135deg, #2563eb, #7c3aed) !important;
    color: white;
}

.assistant-avatar {
    background: linear-gradient(135deg, #06b6d4, #2563eb) !important;
    color: white;
}

.message-bubble-wrap {
    max-width: 72%;
    display: flex;
    flex-direction: column;
}

.user-row .message-bubble-wrap {
    align-items: flex-end;
}

.assistant-row .message-bubble-wrap {
    align-items: flex-start;
}

.message-label {
    font-size: 11.5px;
    font-weight: 900;
    letter-spacing: 0.35px;
    margin-bottom: 5px;
    color: #94a3b8 !important;
}

.user-row .message-label {
    color: #bfdbfe !important;
}

.assistant-row .message-label {
    color: #67e8f9 !important;
}

.chat-bubble {
    width: fit-content;
    max-width: 100%;
    padding: 13px 16px;
    border-radius: 18px;
    line-height: 1.58;
    white-space: pre-wrap;
    font-size: 15px;
    font-weight: 700;
    text-align: left;
}

.user-bubble {
    background: linear-gradient(135deg, #2563eb, #4f46e5, #7c3aed) !important;
    color: #ffffff;
    border-bottom-right-radius: 6px;
    border: 1px solid rgba(191, 219, 254, 0.22);
    box-shadow: 0 8px 18px rgba(79, 70, 229, 0.28);
}

.assistant-bubble {
    background: linear-gradient(135deg, #f8fafc, #e0f2fe) !important;
    color: #0f172a;
    border: 1px solid rgba(125, 211, 252, 0.45);
    border-bottom-left-radius: 6px;
    box-shadow: 0 8px 18px rgba(14, 165, 233, 0.12);
}

/* Textbox */
#message-box textarea {
    border-radius: 18px !important;
    background: rgba(15, 23, 42, 0.96) !important;
    color: #f8fafc !important;
    border: 1px solid rgba(125, 211, 252, 0.52) !important;
    box-shadow: 0 0 14px rgba(14, 165, 233, 0.12);
    font-size: 15.5px !important;
    font-weight: 700 !important;
}

#message-box textarea::placeholder {
    color: #94a3b8 !important;
}

/* Buttons */
#send-btn {
    border-radius: 16px !important;
    background: linear-gradient(135deg, #06b6d4, #2563eb, #7c3aed) !important;
    color: white !important;
    border: none !important;
    font-weight: 900 !important;
    box-shadow: 0 0 18px rgba(37, 99, 235, 0.28);
}

#clear-btn {
    border-radius: 16px !important;
    background: rgba(30, 41, 59, 0.96) !important;
    color: #e2e8f0 !important;
    border: 1px solid rgba(148, 163, 184, 0.35) !important;
    font-weight: 900 !important;
}

/* Examples */
.gr-examples {
    border-radius: 18px !important;
    background: rgba(15, 23, 42, 0.78) !important;
    border: 1px solid rgba(125, 211, 252, 0.25) !important;
    color: #e5e7eb !important;
}

/* Footer */
.footer-text {
    text-align: center;
    color: #bae6fd !important;
    font-size: 13.5px;
    margin-top: 18px;
    font-weight: 800;
}

.status-dot {
    display: inline-block;
    width: 10px;
    height: 10px;
    background: #22c55e;
    border-radius: 50%;
    box-shadow: 0 0 12px #22c55e;
    margin-right: 7px;
}
"""


# ============================================================
# 3. Chat Rendering Functions
# ============================================================

def render_chat(history):
    if history is None or len(history) == 0:
        return """
        <div class="chat-app-card">
            <div class="chat-app-header">
                <div>
                    <div class="chat-app-title">CampusGear AI Chat</div>
                    <div class="chat-app-subtitle">Customer Service Conversation Window</div>
                </div>
                <div class="chat-app-status">
                    <span class="chat-status-dot"></span>
                    Online
                </div>
            </div>
            <div class="chat-display">
                <div class="empty-chat">
                    Start a conversation with CampusGear AI.
                </div>
            </div>
        </div>
        """

    html_parts = [
        """
        <div class="chat-app-card">
            <div class="chat-app-header">
                <div>
                    <div class="chat-app-title">CampusGear AI Chat</div>
                    <div class="chat-app-subtitle">Customer Service Conversation Window</div>
                </div>
                <div class="chat-app-status">
                    <span class="chat-status-dot"></span>
                    Online
                </div>
            </div>
            <div class="chat-display">
        """
    ]

    # reversed(history) + column-reverse keeps the newest message visible.
    for item in reversed(history):
        role = item.get("role", "")
        content = html.escape(str(item.get("content", ""))).replace("\n", "<br>")

        if role == "user":
            html_parts.append(f"""
            <div class="message-row user-row">
                <div class="message-bubble-wrap">
                    <div class="message-label">Customer</div>
                    <div class="chat-bubble user-bubble">{content}</div>
                </div>
                <div class="chat-avatar user-avatar">YOU</div>
            </div>
            """)
        elif role == "assistant":
            html_parts.append(f"""
            <div class="message-row assistant-row">
                <div class="chat-avatar assistant-avatar">AI</div>
                <div class="message-bubble-wrap">
                    <div class="message-label">CampusGear AI</div>
                    <div class="chat-bubble assistant-bubble">{content}</div>
                </div>
            </div>
            """)

    html_parts.append("</div></div>")
    return "\n".join(html_parts)


def user_submit(message, history):
    if history is None:
        history = []

    if message is None or message.strip() == "":
        return "", render_chat(history), history

    try:
        result = campusgear_chatbot(message)
        response = result["chatbot_response"]
    except Exception as e:
        response = f"System error: {str(e)}"

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", render_chat(history), history


def clear_chat():
    empty_history = []
    return "", render_chat(empty_history), empty_history


# ============================================================
# 4. Build Gradio Interface
# ============================================================

with gr.Blocks(css=custom_css) as demo:

    gr.HTML(f"""
    <div id="app-shell">
        <div id="main-title">
            <h1>CampusGear AI Customer Service Chatbot</h1>
            <p>
                An Intelligent Customer Support System Powered by
                LLM, RAG, Fine-Tuned Intent Classification, and a Unified Olist E-Commerce Dataset
            </p>

            <div id="badge-row">
                <span class="ai-badge">LLM Backend</span>
                <span class="ai-badge">RAG Retrieval</span>
                <span class="ai-badge">FAISS Vector Database</span>
                <span class="ai-badge">Fine-Tuned Intent Model</span>
                <span class="ai-badge">Olist Enterprise Dataset</span>
            </div>
        </div>
    </div>
    """)

    gr.HTML("""
    <div class="section-title">
        How to Ask Questions
    </div>
    """)

    gr.HTML(f"""
    <div class="howto-box">
        <b>1. Product recommendation:</b><br>
        Ask about books, stationery, electronics, computer accessories,
        or useful study products.<br>
        Example:
        <code>I am a computer science student. Can you recommend useful study products?</code>

        <br><br>

        <b>2. Shipping and payment questions:</b><br>
        Ask about delivery time, payment methods, online banking,
        credit card, debit card, or e-wallet payment.<br>
        Example:
        <code>What payment methods do you accept?</code>

        <br><br>

        <b>3. Order tracking:</b><br>
        Use an order ID from the CampusGear order database.<br>
        Example:
        <code>Can you track my order {track_order_id}?</code>

        <br><br>

        <b>4. Refund processing:</b><br>
        Ask for refund processing or refund policy using an order ID.<br>
        Example:
        <code>I received a damaged item. I want to refund order {refund_order_id}</code>

        <br><br>

        <b>Available demo order IDs:</b><br>
        {order_tags_html}
    </div>
    """)

    gr.HTML("""
    <div class="section-title">
        <span class="status-dot"></span>
        Live AI Support Chat
    </div>
    """)

    chat_state = gr.State([])

    chat_window = gr.HTML(
        value=render_chat([]),
        label="CampusGear AI Chat Window"
    )

    message = gr.Textbox(
        label="Customer Message",
        placeholder=(
            "Ask about products, shipping, payment methods, orders, "
            f"or refunds. Example: Can you track my order {track_order_id}?"
        ),
        lines=2,
        elem_id="message-box"
    )

    with gr.Row():
        send_btn = gr.Button(
            "Send Message",
            variant="primary",
            elem_id="send-btn"
        )

        clear_btn = gr.Button(
            "Clear Chat",
            elem_id="clear-btn"
        )

    gr.Examples(
        examples=example_questions,
        inputs=message,
        label="Example Customer Questions"
    )

    gr.HTML("""
    <div class="footer-text">
        Developed for MECS1033 Advanced Artificial Intelligence Alternative Assessment
        |
        LLM + RAG + Fine-Tuned Intent Model + Unified Olist Enterprise Dataset
    </div>
    """)

    send_btn.click(
        fn=user_submit,
        inputs=[message, chat_state],
        outputs=[message, chat_window, chat_state]
    )

    message.submit(
        fn=user_submit,
        inputs=[message, chat_state],
        outputs=[message, chat_window, chat_state]
    )

    clear_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[message, chat_window, chat_state]
    )

demo.launch(share=True)

/tmp/ipykernel_12866/2031410104.py:576: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efcf5522c20de1c9df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
